In [16]:
import gymnasium as gym
import numpy as np
from pathlib import Path
import json
import time

# Configuration
NUM_EPISODES = 5000
OUTPUT_DIR = Path("robot_wear_dataset")
OUTPUT_DIR.mkdir(exist_ok=True)

JOINT_NAMES = ["shoulder_pitch", "shoulder_roll", "elbow_flex", 
               "elbow_rot", "wrist_pitch", "wrist_roll", "gripper"]

# Max torque capacity per joint (N·m)
MAX_TORQUE_CAPACITY = np.array([200, 200, 200, 200, 180, 180, 150])

# Test parameters: 3 heights × 6 weights = 18 tests per episode
HEIGHTS = [0.5, 1.0, 1.5]
WEIGHTS = [1, 2, 5, 10, 15, 20]

print("✓ Configuration loaded")
print(f"  Episodes: {NUM_EPISODES}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Tests per episode: {len(HEIGHTS)} × {len(WEIGHTS)} = {len(HEIGHTS) * len(WEIGHTS)}")

✓ Configuration loaded
  Episodes: 5000
  Output: robot_wear_dataset
  Tests per episode: 3 × 6 = 18


In [17]:
def calculate_torque(weight_kg, height_m, joint_index):
    """Calculate stress on joint when lifting weight at height"""
    
    force = weight_kg * 9.8  # Force in Newtons
    base_torque = force * height_m  # Torque = Force × Distance
    
    # Different joints bear different portions of the load
    if joint_index == 6:  # Gripper handles full load
        return base_torque
    elif joint_index in [4, 5]:  # Wrist handles 70%
        return base_torque * 0.7
    elif joint_index in [2, 3]:  # Elbow handles 50%
        return base_torque * 0.5
    else:  # Shoulder handles 25% (supported by body)
        return base_torque * 0.25

print("✓ Torque calculation function defined")

✓ Torque calculation function defined


In [18]:
def calculate_wear_degradation(torques, max_capacities):
    """Calculate wear increase for each joint based on stress"""
    
    wear_increase = np.zeros(7)
    
    for joint_idx in range(7):
        avg_stress = np.mean(torques[:, :, joint_idx])  # Average across all 18 tests
        stress_ratio = avg_stress / max_capacities[joint_idx]  # Normalized stress
        
        # Quadratic degradation: high stress causes exponential wear
        base_wear = (stress_ratio ** 2) * 0.001
        
        # Penalty for exceeding capacity
        if avg_stress > max_capacities[joint_idx]:
            base_wear *= 5
        
        wear_increase[joint_idx] = base_wear
    
    return wear_increase

print("✓ Wear degradation function defined")

✓ Wear degradation function defined


In [19]:
def generate_one_episode(episode_id, current_wear, load_factor=1.0):
    """
    Simulate one episode with a given load_factor (workload intensity).

    load_factor < 1.0 → lighter day (fewer heavy lifts)
    load_factor = 1.0 → normal operation
    load_factor > 1.0 → heavy day (heavier loads, more stress)

    All weights are scaled by load_factor, which directly changes
    torque, stress, and therefore wear increase.
    """

    effective_weights = [w * load_factor for w in WEIGHTS]

    torques = np.zeros((len(HEIGHTS), len(effective_weights), 7))
    lift_results = np.zeros((len(HEIGHTS), len(effective_weights)), dtype=bool)
    stress_history = {}

    for h_idx, height in enumerate(HEIGHTS):
        for w_idx, weight in enumerate(effective_weights):
            for joint_idx in range(7):
                torque = calculate_torque(weight, height, joint_idx)
                torques[h_idx, w_idx, joint_idx] = torque

            joint_torques = torques[h_idx, w_idx, :]
            current_capacities = MAX_TORQUE_CAPACITY * (1 - current_wear ** 2)
            success = np.all(joint_torques < current_capacities)
            lift_results[h_idx, w_idx] = success

    for joint_idx in range(7):
        joint_torques = torques[:, :, joint_idx]
        stress_history[f"joint_{joint_idx}_{JOINT_NAMES[joint_idx]}"] = {
            "total_stress": float(np.sum(joint_torques)),
            "avg_stress":   float(np.mean(joint_torques)),
            "max_stress":   float(np.max(joint_torques)),
            "min_stress":   float(np.min(joint_torques)),
            "num_tests": len(HEIGHTS) * len(effective_weights)
        }

    max_stress_idx = np.argmax(torques[:, :, 6])
    stress_history["most_stressful_test"] = [
        float(HEIGHTS[max_stress_idx // len(effective_weights)]),
        float(effective_weights[max_stress_idx % len(effective_weights)])
    ]

    wear_increase = calculate_wear_degradation(torques, MAX_TORQUE_CAPACITY)
    new_wear = np.clip(current_wear + wear_increase, 0, 1.0)

    current_capacities = MAX_TORQUE_CAPACITY * (1 - current_wear ** 2)
    avg_capacity_percent = np.mean(current_capacities / MAX_TORQUE_CAPACITY) * 100

    metadata = {
        "episode_id":             int(episode_id),
        "load_factor":            float(load_factor),           # NEW — saved for reference
        "wear_start":             current_wear.tolist(),
        "wear_end":               new_wear.tolist(),
        "wear_increase":          wear_increase.tolist(),
        "current_capacity_percent": float(avg_capacity_percent),
        "num_tests_passed":       int(np.sum(lift_results)),
        "num_tests_failed":       int(np.sum(~lift_results)),
        "stress_history":         stress_history,
        "replaced_joints":        []
    }

    return torques, lift_results, metadata, new_wear

print("✓ Episode generator updated (variable load_factor)")

✓ Episode generator updated (variable load_factor)


In [20]:
def generate_full_dataset(num_episodes, seed=42):
    """Generate dataset with realistic workload variation across episodes."""

    rng = np.random.default_rng(seed)
    current_wear = np.array([0.01, 0.01, 0.02, 0.01, 0.03, 0.01, 0.05])
    replacement_log = []

    for episode_id in range(num_episodes):
        # Sample a random daily workload (log-normal keeps it positive, centered at 1.0)
        # sigma=0.4 gives a realistic spread: ~10% of days are 2× normal,
        # ~10% are 0.5× normal — like real factory shift variation
        load_factor = float(np.clip(rng.lognormal(mean=0.0, sigma=0.4), 0.3, 2.5))

        torques, lift_results, metadata, new_wear = generate_one_episode(
            episode_id, current_wear, load_factor=load_factor
        )

        # Joint replacement at threshold
        replaced_joints = []
        for joint_idx in range(7):
            if new_wear[joint_idx] >= 0.7:
                replaced_joints.append(JOINT_NAMES[joint_idx])
                new_wear[joint_idx] = 0.01
        metadata["replaced_joints"] = replaced_joints

        if replaced_joints:
            replacement_log.append({"episode": episode_id, "joints_replaced": replaced_joints})

        episode_dir = OUTPUT_DIR / f"episode_{episode_id:05d}"
        episode_dir.mkdir(exist_ok=True)

        np.save(episode_dir / "torques.npy",      torques)
        np.save(episode_dir / "lift_results.npy", lift_results)
        np.save(episode_dir / "wear_state.npy",   new_wear)

        with open(episode_dir / "metadata.json", "w") as f:
            json.dump(metadata, f, indent=2)

        if (episode_id + 1) % 500 == 0:
            print(f"Episode {episode_id+1:5d}/{num_episodes} | "
                  f"Gripper wear: {new_wear[6]:.3f} | "
                  f"Load factor: {load_factor:.2f} | "
                  f"Passed: {metadata['num_tests_passed']}/18 | "
                  f"Replaced: {len(replaced_joints)}")

        current_wear = new_wear

    print(f"\n✓ Dataset generated!  ({num_episodes} episodes, variable workload)")
    print(f"Total replacements: {len(replacement_log)}")
    for entry in replacement_log[:15]:
        print(f"  Episode {entry['episode']}: replaced {', '.join(entry['joints_replaced'])}")

start = time.time()
generate_full_dataset(NUM_EPISODES)
elapsed = time.time() - start
print(f"\nTime: {elapsed:.1f}s  ({NUM_EPISODES/elapsed:.1f} ep/s)")

Episode   500/5000 | Gripper wear: 0.493 | Load factor: 0.55 | Passed: 16/18 | Replaced: 0
Episode  1000/5000 | Gripper wear: 0.241 | Load factor: 1.39 | Passed: 13/18 | Replaced: 0
Episode  1500/5000 | Gripper wear: 0.038 | Load factor: 0.62 | Passed: 17/18 | Replaced: 0
Episode  2000/5000 | Gripper wear: 0.399 | Load factor: 2.42 | Passed: 9/18 | Replaced: 0
Episode  2500/5000 | Gripper wear: 0.166 | Load factor: 1.64 | Passed: 11/18 | Replaced: 0
Episode  3000/5000 | Gripper wear: 0.022 | Load factor: 0.87 | Passed: 15/18 | Replaced: 0
Episode  3500/5000 | Gripper wear: 0.450 | Load factor: 1.27 | Passed: 11/18 | Replaced: 0
Episode  4000/5000 | Gripper wear: 0.216 | Load factor: 0.61 | Passed: 17/18 | Replaced: 0
Episode  4500/5000 | Gripper wear: 0.020 | Load factor: 0.82 | Passed: 15/18 | Replaced: 0
Episode  5000/5000 | Gripper wear: 0.449 | Load factor: 0.70 | Passed: 15/18 | Replaced: 0

✓ Dataset generated!  (5000 episodes, variable workload)
Total replacements: 8
  Episode 7